# GPS and Route Reliability Analysis

I built this notebook as a public, reproducible version of the GPS-quality branch of my Spring MSBA exploratory analytics live case. It uses synthetic routes and telemetry so that I can demonstrate the analytical workflow without exposing client data.

## Decision question

When a route looks incomplete, is the operation actually underperforming—or is the GPS device producing an unreliable record? I separate those possibilities through data validation, temporal coverage, spatial movement, geofence completion, route order, and device-level recurrence.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from generate_sample_data import generate
from gps_quality import (
    build_device_health,
    build_stop_validation,
    build_trip_quality,
    summarize_route_validation,
)

## 1. Load the public synthetic data

The public fixture contains 60 trips, GPS pings from six devices and two providers, and three planned routes with six stops each. Known failure modes are injected deterministically so the validation logic can be tested.

In [ ]:
trips, positions, route_stops = generate()

pd.DataFrame({
    'dataset': ['trips', 'positions', 'route_stops'],
    'rows': [len(trips), len(positions), len(route_stops)],
    'unique_trips': [trips.trip_id.nunique(), positions.trip_id.nunique(), None],
})

## 2. Measure temporal and spatial GPS quality

For every trip, I compare observed pings with the provider-specific polling expectation, measure the longest silence, detect frozen coordinates, calculate path distance with vectorized Haversine distance, and flag coordinate jumps that imply road speeds above 126 km/h.

In [ ]:
trip_quality = build_trip_quality(trips, positions)
trip_quality[[
    'trip_id', 'tracking_device_id', 'ping_coverage_pct', 'max_gap_s',
    'path_distance_km', 'max_speed_kph', 'flag_frozen',
    'flag_implausible_speed', 'quality_score', 'quality_label',
]].head(10)

## 3. Validate planned stops with geofences

I match each GPS trace to its planned stops with a 175-metre Haversine geofence. This produces stop-level minimum distance, trip-level completion, and an order score based on the sequence of first geofence visits.

In [ ]:
stop_validation = build_stop_validation(trips, positions, route_stops)
route_validation = summarize_route_validation(stop_validation)
route_validation.head(10)

## 4. Separate trip anomalies from recurring device issues

A single bad trip may be noise. Repeated failures on one tracking device are operationally actionable, so I roll trip diagnostics into a device-health report.

In [ ]:
scored_trips = trip_quality.merge(route_validation, on=['trip_id', 'route_id'])
device_health = build_device_health(scored_trips)
device_health.sort_values('mean_quality_score')

## What the controlled test demonstrates

The pipeline distinguishes four different failure patterns instead of treating every incomplete trace as the same problem: low polling coverage, a long reporting gap, frozen coordinates, and an impossible spatial jump. Healthy devices retain full stop completion and plausible movement, while the frozen device records only its starting geofence.

This notebook demonstrates the public method, not the confidential live-case findings. Thresholds are explicit and should be calibrated to a real provider's polling cadence, vehicle type, road context, and operating policy before production use.